# 02 · Train and evaluate one configuration

One run = **one model × one strategy × one fold**. Set the three values in *Parameters*, then *Run all*.
Everyone uses this notebook on their own Kaggle account, so all 65 cross-validation runs share
the same code, the same frozen split (`data/splits/folds.csv`) and the same settings
(`configs/default.yaml`, seed 42, 224 × 224).

What we test: how much of an ImageNet backbone must be retrained to grade diabetic retinopathy.
*Feature extraction* trains only the new 5-grade head, *partial fine-tuning* also retrains the last
third of the backbone, *full fine-tuning* retrains everything; the small CNN trained from scratch
shows what we get without any transfer.

**Kaggle settings:** Accelerator GPU (T4 or P100) · Internet on (repository + ImageNet weights) ·
Input: dataset `jockeroika/diabetic-retinopathy`. Weights, logs and resumable state are saved in
`/kaggle/working/outputs/` (the notebook's Outputs) and never go to Git.

In [ ]:
# Locate the repository, or clone the branch to test when this runs on Kaggle/Colab.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/MOUATEAlaaeddine/dr-transfer-learning.git"
BRANCH = "main"          # set to a feature branch only to test a pull request

here = Path.cwd()
if (here / "src" / "drtl").exists():
    root = here
elif (here.parent / "src" / "drtl").exists():          # opened from notebooks/
    root = here.parent
else:                                                    # fresh cloud runtime
    if not Path("dr-transfer-learning").exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL], check=True)
    root = Path("dr-transfer-learning").resolve()

os.chdir(root)
sys.path.insert(0, str(root / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Repository:", root, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())

## Parameters

| Member | `MODEL` | `STRATEGY` |
|---|---|---|
| Alaa Eddine | `vit_b16` | `feature_extraction`, `partial_finetuning`, `full_finetuning` |
| Abdelati | `scratch_cnn` · `vgg19` | `scratch` · the three transfer strategies |
| Ikram | `densenet201` | the three transfer strategies |
| Youssef | `mobilenetv4` | the three transfer strategies |

`FOLD` goes from 0 to 4. Start with fold 0 and check the outputs before launching the others.
`SMOKE = True` trains 2 epochs on 4 batches to check the plumbing (never a result).
`SYNTHETIC = True` uses SIMULATED images instead of the dataset (pipeline test only).
If VGG19 or ViT runs out of GPU memory, set `OVERRIDES = {"training": {"batch_size": 16, "accumulation_steps": 2}}`;
the override is written in the run record.

In [ ]:
MODEL = "mobilenetv4"            # mobilenetv4 · densenet201 · vgg19 · vit_b16 · scratch_cnn
STRATEGY = "feature_extraction"   # feature_extraction · partial_finetuning · full_finetuning · scratch
FOLD = 0                          # 0 … 4
SMOKE = False
SYNTHETIC = False
RESUME = True                     # continue from outputs/<run_id>/last.pt after an interrupted session
RESUME_FROM = ""                  # Kaggle: previous version's outputs added as Input, e.g. "/kaggle/input/<notebook>/outputs"
OVERRIDES = {}

## The configuration we are about to train

Learning rates come from the shared config: the new head starts random and learns at 1e-3; the
pretrained backbone only needs small corrections (1e-4 partial, 3e-5 full). Frozen BatchNorm layers
stay in evaluation mode so their ImageNet statistics do not drift.

In [ ]:
import pandas as pd
from drtl.config import load_config
from drtl.models import build_for_run, count_parameters, strategy_settings

config = load_config(overrides=OVERRIDES)
counts = count_parameters(build_for_run(config, MODEL, STRATEGY, pretrained=False))
pd.Series({"model": MODEL, "timm name": config["models"][MODEL], "strategy": STRATEGY, "fold": FOLD,
           **strategy_settings(config, MODEL, STRATEGY),
           "trainable / total": f"{counts['trainable']:,} / {counts['total']:,}"}).to_frame("value")

## Train

Training folds: the four other folds of the 85 % CV part. Validation: fold `FOLD`. The 15 % test set
is never loaded here; it is used once, at the end, for the selected configurations (Ikram, Step 7).
Loss weights are computed from the training folds only. The kept checkpoint is the epoch with the
best **validation macro F1**; training stops when validation loss has not improved for 5 epochs.

In [ ]:
import shutil
from drtl.run import run_experiment

if RESUME_FROM:   # bring last.pt of an interrupted Kaggle version back into this session
    shutil.copytree(RESUME_FROM, "/kaggle/working/outputs", dirs_exist_ok=True)
record = run_experiment(MODEL, STRATEGY, FOLD, overrides=OVERRIDES, smoke=SMOKE,
                        synthetic=SYNTHETIC, resume=RESUME)

## Result of this run

Validation metrics of the best epoch. They describe one fold only; the comparison uses the mean ±
standard deviation over the five folds (`python -m drtl.results`, then Ikram's evaluation).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

if record["status"] != "completed":
    print(f"⚠ {record['status'].upper()} run on {record['data_source']}: SIMULATED / plumbing check, not a result.")
keys = ["run_id", "status", "data_source", "best_epoch", "epochs_run", "duration_s", "sec_per_epoch",
        "val_accuracy", "val_precision_macro", "val_recall_macro", "val_f1_macro", "params_trainable",
        "config_hash", "splits_hash", "git_commit", "checkpoint_ref"]
display(pd.Series({k: record[k] for k in keys}).to_frame("value"))

run_dir = Path(record["checkpoint_ref"]).parent
history = pd.read_csv(run_dir / "history.csv")
preds = pd.read_csv(record["predictions_path"])
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
history.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], title="Loss")
history.plot(x="epoch", y="val_f1_macro", ax=axes[1], title="Validation macro F1", legend=False)
axes[1].axvline(record["best_epoch"], ls="--", c="grey")
ConfusionMatrixDisplay.from_predictions(preds["label"], preds["pred"], labels=range(5),
    display_labels=config["data"]["class_names"], ax=axes[2], colorbar=False, xticks_rotation=30)
axes[2].set_title(f"Validation fold {FOLD} (best epoch)")
title = record["run_id"] + ("  ·  SIMULATED" if record["status"] != "completed" else "")
fig.suptitle(title); fig.tight_layout(); fig.savefig(run_dir / "curves.png", dpi=120)

## What to keep

* **Kaggle Outputs** (`/kaggle/working/outputs/<run_id>/`): `best.pt`, `last.pt`, `history.csv`, `curves.png`.
  Never commit them.
* **Pull request**: the two small files `results/runs/<run_id>.json` and `<run_id>_val_predictions.csv`,
  zipped below as `results_<run_id>.zip`. Download it from the Output tab, unzip it into `results/runs/`
  of your branch and commit with `exp: <run_id>`. Do not edit `results/runs.csv` by hand.

In [ ]:
import shutil
if record["status"] == "completed":
    files = [Path(record["record_file"]), Path(record["predictions_path"])]
    bundle = run_dir.parents[1] / f"results_{record['run_id']}"
    bundle.mkdir(exist_ok=True)
    for f in files:
        shutil.copy(f, bundle)
    print("Download:", shutil.make_archive(str(bundle), "zip", bundle))
else:
    print("Smoke/SIMULATED run: nothing to commit.")
print("Checkpoint:", record["checkpoint_ref"])